## Path setups and imports

In [10]:
# Make src/ importable
import sys
from pathlib import Path

SRC = Path.cwd().parent / "src"
sys.path.insert(0, str(SRC))

import config
import models
import evaluation
import crossval

import numpy as np
import pandas as pd
import torch
import joblib
from sklearn.utils.class_weight import compute_class_weight

print("Imports OK: Project root:", config.PROJECT_ROOT)

Imports OK: Project root: /home/koala/lab/adversec


## Load artifacts and detect the GPU

In [2]:
# Load the preprocessed arrays saved at the end of Stage 2
data = np.load(config.PROCESSED_DIR / "stage2_arrays.npz")
X_train = data["X_train"]
y_train = data["y_train"]
X_test = data["X_test"]
y_test = data["y_test"]

# Load the fitted encoder to translate class integers back to names.
label_encoder = joblib.load(config.PROCESSED_DIR / "label_encoder.joblib")
class_names = list(label_encoder.classes_)

# Pick GPU if available
device = "cuda" if torch.cuda.is_available() else "CPU"

print(f"X_train : {X_train.shape}    X_test : {X_test.shape}")
print(f"classes : {class_names}")
print(f"device  : {device}")

X_train : (3838, 9)    X_test : (718, 9)
classes : ['DoS', 'benign', 'spoofing-GAS', 'spoofing-RPM', 'spoofing-SPEED', 'spoofing-STEERING_WHEEL']
device  : cuda


## Compute class weights

In [3]:
# Compute a weight per class inversely propotional to its frequency in the training labels.
classes_arr = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes_arr,
    y=y_train,
)

# Show the weight each class gets
print("Class weights:")
for cls_int, w in zip(classes_arr, weights):
    print(f"    {cls_int} ({class_names[cls_int]:24s}): {w:.3f}")

# Conver to a torch tensor on the device for the CNNs loss function later
class_weights_tensor = torch.tensor(weights, dtype=torch.float32, device=device)

Class weights:
    0 (DoS                     ): 3.198
    1 (benign                  ): 0.225
    2 (spoofing-GAS            ): 3.198
    3 (spoofing-RPM            ): 3.198
    4 (spoofing-SPEED          ): 3.198
    5 (spoofing-STEERING_WHEEL ): 3.198


## Train and evaluate RF

In [4]:
# Build the RF
rf = models.build_random_forest(random_seed=config.RANDOM_SEED)

# Train it
rf.fit(X_train, y_train)

# Predict on the held-out test set
rf_pred = rf.predict(X_test)

# Score it throug the shared evaluation harness
rf_results = evaluation.evaluate_model(
    y_test, rf_pred, class_names, model_name="Random Forest (baseline)"
)


    Random Forest (baseline)
    Accuracy    : 0.9972
    Macro-F1    : 0.7761

   Per-class report:
                         precision    recall  f1-score   support

                    DoS       1.00      0.75      0.86         4
                 benign       1.00      1.00      1.00       709
           spoofing-GAS       0.00      0.00      0.00         1
           spoofing-RPM       0.67      1.00      0.80         2
         spoofing-SPEED       1.00      1.00      1.00         1
spoofing-STEERING_WHEEL       1.00      1.00      1.00         1

               accuracy                           1.00       718
              macro avg       0.78      0.79      0.78       718
           weighted avg       1.00      1.00      1.00       718

     Confusion matrix (rows=true, cols=pred):
[[  3   0   0   1   0   0]
 [  0 709   0   0   0   0]
 [  0   1   0   0   0   0]
 [  0   0   0   2   0   0]
 [  0   0   0   0   1   0]
 [  0   0   0   0   0   1]]


## Train the 1D-CNN

In [5]:
# Build a fresh CNN
cnn = models.CNN1D(n_features=9, n_classes=6)

# Train it.
cnn = models.train_cnn(
    cnn,
    X_train, y_train,
    n_epochs=50,
    batch_size=64,
    lr = 1e-3,
    class_weights = class_weights_tensor,
    device=device,
    random_seed = config.RANDOM_SEED,
)

    epoch   1/50     loss 1.6394
    epoch   5/50     loss 0.2051
    epoch  10/50     loss 0.0129
    epoch  15/50     loss 0.0039
    epoch  20/50     loss 0.0024
    epoch  25/50     loss 0.0018
    epoch  30/50     loss 0.0015
    epoch  35/50     loss 0.0013
    epoch  40/50     loss 0.0014
    epoch  45/50     loss 0.0011
    epoch  50/50     loss 0.0011


## Evaluate the CNN on the test set

In [6]:
# Switch the model to evaluation mode
cnn.eval()
with torch.no_grad():
    X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)
    logits = cnn(X_test_t)
    cnn_pred = logits.argmax(dim=1).cpu().numpy()

# Score throug the harness
cnn_results = evaluation.evaluate_model(
    y_test, cnn_pred, class_names, model_name="1D=CNN (baseline)"
)


    1D=CNN (baseline)
    Accuracy    : 0.9930
    Macro-F1    : 0.6606

   Per-class report:
                         precision    recall  f1-score   support

                    DoS       0.67      1.00      0.80         4
                 benign       1.00      1.00      1.00       709
           spoofing-GAS       1.00      1.00      1.00         1
           spoofing-RPM       0.50      0.50      0.50         2
         spoofing-SPEED       0.50      1.00      0.67         1
spoofing-STEERING_WHEEL       0.00      0.00      0.00         1

               accuracy                           0.99       718
              macro avg       0.61      0.75      0.66       718
           weighted avg       0.99      0.99      0.99       718

     Confusion matrix (rows=true, cols=pred):
[[  4   0   0   0   0   0]
 [  1 706   0   1   1   0]
 [  0   0   1   0   0   0]
 [  0   1   0   1   0   0]
 [  0   0   0   0   1   0]
 [  1   0   0   0   0   0]]


# Scheme B

In [7]:
# Scheme B: stratified 5-fold over the duplicated rows. Leaks copies across folds by design.
# This is the inflated contrast, not a truth result.

print("SCHEME B: row-level CV (leaky, inflated contrast)\n")

results_B = crossval.crossval_rowlevel(
    X_train, y_train,
    n_splits=5,
    device=device,
    random_seed=config.RANDOM_SEED,
    cnn_epochs=50,
    class_weights=class_weights_tensor,
)

print("\nScheme B summary (macro-F1):")
print(f"    RF  : {np.mean(results_B["rf"]):.4f} ± {np.std(results_B["rf"]):.4f}")
print(f"    CNN  : {np.mean(results_B["cnn"]):.4f} ± {np.std(results_B["cnn"]):.4f}")

SCHEME B: row-level CV (leaky, inflated contrast)

    epoch   1/50     loss 1.6907
    epoch   5/50     loss 0.2245
    epoch  10/50     loss 0.0149
    epoch  15/50     loss 0.0047
    epoch  20/50     loss 0.0029
    epoch  25/50     loss 0.0022
    epoch  30/50     loss 0.0016
    epoch  35/50     loss 0.0012
    epoch  40/50     loss 0.0012
    epoch  45/50     loss 0.0011
    epoch  50/50     loss 0.0011
    fold 1: RF=1.0000     CNN=0.9935
    epoch   1/50     loss 1.6948
    epoch   5/50     loss 0.1822
    epoch  10/50     loss 0.0168
    epoch  15/50     loss 0.0045
    epoch  20/50     loss 0.0027
    epoch  25/50     loss 0.0023
    epoch  30/50     loss 0.0022
    epoch  35/50     loss 0.0015
    epoch  40/50     loss 0.0016
    epoch  45/50     loss 0.0013
    epoch  50/50     loss 0.0012
    fold 2: RF=1.0000     CNN=1.0000
    epoch   1/50     loss 1.6778
    epoch   5/50     loss 0.2239
    epoch  10/50     loss 0.0203
    epoch  15/50     loss 0.0062
    epoch  20/50 

## Scheme A

In [11]:
# Reload the un-duplicated signatures
strict = pd.read_csv(config.PROCESSED_DIR / "ciciov2024_strict.csv")

# Scheme A: 2-folds
# Splits signatures first, duplicates inside each train fold only - no leakage.
print ("SCHEME A: signature-level CV (no leakage)\n")
results_A = crossval.crossval_signature_level(
    strict, config.FEATURE_COLUMNS,
    n_splits=2,
    dup_target=200,
    device=device,
    random_seed=config.RANDOM_SEED,
    cnn_epochs=50,
)

print("\nScheme A summary (macro-F1):")
print(f"    RF  : {np.mean(results_A["rf"]):.4f} ± {np.std(results_A["rf"]):.4f}")
print(f"    CNN  : {np.mean(results_A["cnn"]):.4f} ± {np.std(results_A["cnn"]):.4f}")

SCHEME A: signature-level CV (no leakage)

Label mapping:
    0 -> DoS
    1 -> benign
    2 -> spoofing-GAS
    3 -> spoofing-RPM
    4 -> spoofing-SPEED
    5 -> spoofing-STEERING_WHEEL
    epoch   1/50     loss 1.2926
    epoch   5/50     loss 0.4407
    epoch  10/50     loss 0.0193
    epoch  15/50     loss 0.0048
    epoch  20/50     loss 0.0023
    epoch  25/50     loss 0.0017
    epoch  30/50     loss 0.0016
    epoch  35/50     loss 0.0009
    epoch  40/50     loss 0.0007
    epoch  45/50     loss 0.0008
    epoch  50/50     loss 0.0007
    fold 1: RF=0.6466     CNN=0.7218
Label mapping:
    0 -> DoS
    1 -> benign
    2 -> spoofing-GAS
    3 -> spoofing-RPM
    4 -> spoofing-SPEED
    5 -> spoofing-STEERING_WHEEL
    epoch   1/50     loss 1.2867
    epoch   5/50     loss 0.2218
    epoch  10/50     loss 0.0153
    epoch  15/50     loss 0.0046
    epoch  20/50     loss 0.0020
    epoch  25/50     loss 0.0014
    epoch  30/50     loss 0.0010
    epoch  35/50     loss 0.0006
   

## Stage 3 Summary: Baselines and the Accuracy Trap, Quantified

Trained Random Forest and 1D-CNN baselines, then cross-validated under two schemes.

### Cross-validation results (macro-F1)

| Scheme | Random Forest | 1D-CNN |
|---|---|---|
| **A — signature-level (honest)** | 0.676 ± 0.030 | **0.774 ± 0.052** |
| **B — row-level (leaky contrast)** | 1.000 ± 0.000 | 0.997 ± 0.003 |

### What this shows

1. **The accuracy trap, measured on our own pipeline.** Letting duplicate copies
   leak across folds (Scheme B) lifts macro-F1 from ~0.68–0.77 to ~1.00 with *no new
   information* — only leakage. This demonstrates, with a number, why prior work
   reports near-perfect scores.

2. **The single split was misleading.** A single split suggested RF (0.776) beat the
   CNN (0.661). Honest cross-validation reverses this: the CNN (0.774) slightly leads
   the RF (0.676) — opposite to both the single split and the original hypothesis.

3. **No confident winner at baseline.** The standard deviations overlap, and with
   attack classes of only 2–5 signatures, two folds cannot support a confident
   ranking. The honest conclusion is that the two models are **not clearly separable**
   on de-duplicated data, with a slight edge to the CNN. This challenges the claim that
   Random Forest is the better choice in this setting.

The live question now moves to **robustness under adversarial attack** (Stage 4), where
the architectures are expected to diverge meaningfully.